# Feature Extraction (Sliding Window)

##  从加速度数据中提取 119 个手工特征用于传统机器学习

本代码对经过滚动窗口计算（静态/动态分量）后的预处理数据，使用滑动窗口（窗口 50，步长 25）提取每个时间窗的 119 个手工特征（包括均值、方差、频域主频及幅值、俯仰角、横滚角、ODBA 等）。特征列表由 119-features.yaml 配置文件定义。该步骤对应论文第 2.6 节及附表 S6，为传统机器学习模型（LightGBM / XGBoost）提供输入，以与深度学习模型进行性能对比。

## Import Modules

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from omegaconf import OmegaConf

import sys
sys.path.append("../")  # parent dir so `from src import ...` works
sys.dont_write_bytecode = True
%load_ext autoreload
%autoreload 2

from src import feature_extraction
from src import utils

C:\Users\bxy11\Desktop\dl-wabc\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
debug_test_mode = False
# debug_test_mode = True  # 取消注释可启用调试模式（跳过特征提取和保存）

# species_list = ["omizunagidori", "umineko"]  # 同时处理两个物种（全量运行）
species_list = ["omizunagidori"]               # 仅处理白额鹱（当前使用）
# species_list = ["umineko"]                   # 仅处理黑尾鸥

data_root_dir = "../data/datasets/logbot_data"
feature_config_path = "../configs/features/119-features.yaml"

WINDOW_SIZE = 50          # 滑动窗口长度：50 个采样点（2 秒）
WINDOW_STEPSIZE = 25      # 滑动步长：25 个采样点（1 秒，50% 重叠）

## Load feature list & build output column names

In [3]:
# 步骤 1：加载特征配置文件
config = OmegaConf.load(feature_config_path)

# =============================================================================
# 步骤 2：提取特征名称列表
# config.features_list 从 YAML 配置中读取特征名称列表
# 例如：['acc_x_mean', 'acc_x_var', 'acc_x_max', ...]
# len(features) 打印特征总数（应为 119）
# =============================================================================
features = config.features_list
print(f"Number of features: {len(features)}")

# =============================================================================
# 步骤 3：构建输出 DataFrame 的列名列表
# col_names 定义了特征提取后 CSV 文件的列结构：
# 前 4 列（元数据列）：
#   - 'animal_id'：个体 ID（如 OM1803）
#   - 'unixtime'：Unix 时间戳（用于时间对齐）
#   - 'label'：行为标签（如 "stationary"）
#   - 'label_id'：行为标签编号（0-5，用于模型训练）
# =============================================================================
col_names = ['animal_id', 'unixtime', 'label', 'label_id'] + list(features)
print(col_names)

Number of features: 119
['animal_id', 'unixtime', 'label', 'label_id', 'acc_x_mean', 'acc_y_mean', 'acc_z_mean', 'acc_x_var', 'acc_y_var', 'acc_z_var', 'acc_x_std', 'acc_y_std', 'acc_z_std', 'acc_x_cv', 'acc_y_cv', 'acc_z_cv', 'acc_x_skew', 'acc_y_skew', 'acc_z_skew', 'acc_x_kurtosis', 'acc_y_kurtosis', 'acc_z_kurtosis', 'acc_x_max', 'acc_y_max', 'acc_z_max', 'acc_x_min', 'acc_y_min', 'acc_z_min', 'acc_x_range', 'acc_y_range', 'acc_z_range', 'acc_x_q25', 'acc_y_q25', 'acc_z_q25', 'acc_x_q50', 'acc_y_q50', 'acc_z_q50', 'acc_x_q75', 'acc_y_q75', 'acc_z_q75', 'mag_mean', 'mag_var', 'mag_skew', 'mag_kurtosis', 'mag_rms', 'acc_x_norm', 'acc_y_norm', 'acc_z_norm', 'acc_x_ac', 'acc_y_ac', 'acc_z_ac', 'acc_x_trend', 'acc_y_trend', 'acc_z_trend', 'mag_trend', 'cov_xy', 'cov_yz', 'cov_zx', 'corr_xy', 'corr_yz', 'corr_zx', 'diff_xy_mean', 'diff_yz_mean', 'diff_zx_mean', 'diff_xy_std', 'diff_yz_std', 'diff_zx_std', 'acc_x_st_mean', 'acc_y_st_mean', 'acc_z_st_mean', 'acc_x_st_var', 'acc_y_st_var', 

## Run feature extraction per species / per individual

In [4]:
# 打印特征提取开始信息
print("-----------------------------------")
print("running feature extraction !")
print("-----------------------------------")

# 外层循环：遍历物种列表
for species in species_list:

    # --- 步骤 1：获取包含静态/动态分量的数据文件列表 ---
    # target 匹配 data_after_rolling_calc 目录下的所有 CSV 文件
    target = f"{data_root_dir}/feature_extraction/data_after_rolling_calc/{species}/*.csv"
    labelled_data_path_list = sorted(glob.glob(target))
    print(f"species: {species} | N of individuals: {len(labelled_data_path_list)}")

    # --- 步骤 2：遍历每个个体 ---
    for labelled_data_path in labelled_data_path_list:
        start_time_perf, start_time_process = utils.start_time_counter()
        print(f"labelled_data_path: {labelled_data_path}")
        print("--------------------------------------")

        # --- 步骤 2a：获取个体 ID ---
        # 从文件路径中提取文件名，去掉 .csv 后缀
        animal_id = os.path.basename(labelled_data_path).replace(".csv", "")
        print(f"loading csv file for {animal_id} ...")

        # --- 步骤 2b：使用 pandas 读取数据 ---
        # low_memory=False：避免 pandas 在读取大文件时进行内存优化（提高速度）
        # reset_index(drop=True)：重置索引，丢弃原有索引
        df = pd.read_csv(labelled_data_path, low_memory=False).reset_index(drop=True)
        print(f"length of the input df: {len(df)}")
        utils.end_time(start_time_perf, start_time_process)

        # --- 步骤 2c：可选的交叉验证检查（与 NPZ 扫描文件对比）---
        # 对应论文第 2.1 节：
        #   - 确保特征提取与数据预处理（NPZ 提取）的一致性
        #   - 验证滑动窗口提取是否正确
        #
        # scan_path：指向 npz_files_scan 目录下的扫描文件
        # 如果存在，则用于验证窗口提取是否正确
        scan_path = f"{data_root_dir}/npz_files_scan/{species}/df_scan_{animal_id}.csv"
        if os.path.exists(scan_path):
            df_scan = pd.read_csv(scan_path)
            scanned_unixtime_values = df_scan["unixtime"].values
            scanned_label_id_values = df_scan["label_id"].values
            do_scan_check = True
        else:
            print(f"df_scan file not found at {scan_path} -> skipping cross-check with npz pipeline")
            scanned_unixtime_values = np.array([])
            scanned_label_id_values = np.array([])
            do_scan_check = False

        # --- 步骤 2d：构建窗口级别的扫描参考 ---
        # 提取每个窗口的最后一个采样点作为该窗口的代表
        # 用于后续的交叉验证检查
        scanned_unixtime_list = []
        scanned_label_id_list = []
        for i in range(0, scanned_label_id_values.size):
            if (i + 1) % WINDOW_SIZE == 0:
                scanned_unixtime_list.append(scanned_unixtime_values[i])
                scanned_label_id_list.append(scanned_label_id_values[i])

        # --- 步骤 2e：滑动窗口特征提取 ---
        # 初始化变量：
        #   - window_counter：有效窗口计数
        #   - df_values：DataFrame 的 NumPy 数组表示（提高性能）
        #   - data_array_list：存储每个窗口的特征向量
        #   - label_id_list：存储每个窗口的标签（用于检查）
        window_counter = 0
        df_values = df.values
        data_array_list = []
        label_id_list = []

        start_time_perf, start_time_process = utils.start_time_counter()

        # --- 步骤 2f：遍历所有窗口 ---
        # tqdm 显示进度条，便于监控处理进度
        # range(0, len(df) - WINDOW_SIZE, WINDOW_STEPSIZE)
        #   - 起始：0
        #   - 结束：len(df) - WINDOW_SIZE（最后一个完整窗口）
        #   - 步长：WINDOW_STEPSIZE（25，即 50% 重叠）
        for i in tqdm(range(0, len(df) - WINDOW_SIZE, WINDOW_STEPSIZE)):

            # --- 提取窗口数据 ---
            window_tmp = df_values[i:i + WINDOW_SIZE]

            # 提取时间戳（列索引 1）
            unixtime_tmp = window_tmp[:, 1].astype(np.float64)
            unixtime_tmp_representative = np.float64(unixtime_tmp[-1])

            # --- 检查时间连续性 ---
            # unixtime_diff >= 2.0：窗口内时间跨度超过 2 秒
            # 说明窗口内存在时间戳断裂，跳过该窗口
            unixtime_diff = np.max(unixtime_tmp) - np.min(unixtime_tmp)
            if unixtime_diff >= 2.0:
                continue  # 跳过时间不连续的窗口

            # --- 提取加速度数据 ---
            X_tmp = window_tmp[:, 2:5].astype(np.float64)  # acc_x, acc_y, acc_z

            # --- 检查数据质量 ---
            num_zeros_in_X = np.sum(X_tmp == 0)  # 零值数量
            label_id_tmp = window_tmp[:, 6].astype(np.float64)  # label_id
            num_na_in_label_id = np.sum(np.isnan(label_id_tmp))  # NaN 数量
            num_unique_label_id = len(np.unique(label_id_tmp))  # 唯一标签数

            # --- 窗口有效性条件（必须全部满足）---
            # 1. num_zeros_in_X < 5：加速度数据中零值少于 5 个（避免设备异常数据）
            # 2. num_na_in_label_id == 0：标签列没有 NaN（所有采样点都有标签）
            # 3. num_unique_label_id == 1：窗口内所有标签一致（纯行为窗口）
            if not (num_zeros_in_X < 5 and num_na_in_label_id == 0 and num_unique_label_id == 1):
                continue

            # --- 提取窗口的元数据 ---
            unixtime = unixtime_tmp[-1]          # 窗口最后一个采样点的时间戳
            label = window_tmp[:, 5][-1]         # 窗口最后一个采样点的标签
            label_id = int(label_id_tmp[-1])     # 窗口最后一个采样点的标签 ID

            label_id_list.append(label_id)       # 收集标签（用于检查）
            window_counter += 1

            # --- 调试模式：跳过特征计算 ---
            if debug_test_mode == True:
                continue

            # --- 计算 119 个手工特征 ---
            # 调用 feature_extraction.calc_features_for_one_sliding_window()
            # 输入：一个窗口的原始数据
            # 输出：119 个特征值的列表
            feature_list = feature_extraction.calc_features_for_one_sliding_window(window_tmp)

            # --- 构建特征向量 ---
            # data_array_1：元数据 [animal_id, unixtime, label, label_id]
            # data_array_2：119 个特征值
            # data_array_tmp：拼接后的完整特征向量（123 个元素）
            data_array_1 = np.array([animal_id, unixtime, label, label_id])
            data_array_2 = np.array(feature_list)
            data_array_tmp = np.concatenate([data_array_1, data_array_2], axis=0)
            data_array_list.append(data_array_tmp)

            # --- 交叉验证检查 ---
            # 验证当前窗口的代表性时间戳是否在扫描参考中
            # 确保特征提取与 NPZ 提取的窗口一致
            if do_scan_check and unixtime_tmp_representative not in scanned_unixtime_list:
                raise Exception(
                    f"unixtime {unixtime_tmp_representative} not included in scanned_unixtime_values"
                )

        # --- 步骤 2g：长度/一致性检查 ---
        print(f"window_counter: {window_counter}")
        print(f"len(label_id_list): {len(label_id_list)}")
        print(f"len(scanned_unixtime_list): {len(scanned_unixtime_list)}")
        print(f"len(scanned_label_id_list): {len(scanned_label_id_list)}")
        print(f"scanned_unixtime_values.shape: {scanned_unixtime_values.shape}")
        print(f"scanned_label_id_values.shape: {scanned_label_id_values.shape}")

        data_array = np.asarray(data_array_list)
        print(f"data_array.shape: {data_array.shape}")

        # --- 步骤 2h：标签检查（与扫描参考对比）---
        print("----- Check label_id and scanned label_id              -----")
        if do_scan_check:
            label_check_array = np.array(label_id_list) - np.array(scanned_label_id_list)
            print(f"np.sum(label_check_array): {np.sum(label_check_array)}")
            is_array_equal = np.array_equal(np.array(label_id_list), np.array(scanned_label_id_list))
            print(f"is_array_equal: {is_array_equal}")

            # 检查窗口数量是否一致
            if np.array(label_id_list).shape[0] != np.array(scanned_label_id_list).shape[0]:
                raise Exception(
                    f"extracted window size {np.array(label_id_list).shape[0]} and "
                    f"scanned sample size {np.array(scanned_label_id_list).shape[0]} are different!"
                )

            # 检查标签是否完全匹配
            if np.sum(label_check_array) > 0:
                raise Exception(
                    f"np.sum(label_check_array) should be 0, but {np.sum(label_check_array)}"
                )
        else:
            print("skipped (df_scan not available)")

        # --- 步骤 2i：保存特征数据 ---
        if debug_test_mode:
            print("| debug test mode -> do not save extracted features |")
            continue

        # 创建 DataFrame
        # col_names：4 个元数据列 + 119 个特征列
        df_features = pd.DataFrame(data=data_array, columns=col_names).reset_index(drop=True)

        # --- 转换标签格式 ---
        # utils.convert_pandas_labels() 将标签转换为标准格式
        # 对应论文第 2.6 节：确保标签与模型训练时的格式一致
        if species == "omizunagidori":
            label_species = "om"
        elif species == "umineko":
            label_species = "um"
        df_features = utils.convert_pandas_labels(df_features, label_species)

        print(f"N of extracted windows (len(df_features)): {len(df_features)}")
        print(f"N of extracted window counter: {window_counter}")
        print(df_features.head(3))

        # --- 保存为 CSV ---
        # 输出路径：data/datasets/logbot_data/feature_extraction/acc_features/{species}/{animal_id}.csv
        base_dir = f"{data_root_dir}/feature_extraction/acc_features"
        save_dir = f"{base_dir}/{species}"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/{animal_id}.csv"
        df_features.to_csv(save_path, index=False)
        print(f"df_features save at path: {save_path}")

        utils.end_time(start_time_perf, start_time_process)

    # --- 步骤 3：物种处理完成 ---
    print(f"All df_features for {species} were successfully saved.")

-----------------------------------
running feature extraction !
-----------------------------------
species: omizunagidori | N of individuals: 28
labelled_data_path: ../data/datasets/logbot_data/feature_extraction/data_after_rolling_calc/omizunagidori\OM1803.csv
--------------------------------------
loading csv file for OM1803 ...
length of the input df: 1008000
elapsed_time_perf:  00:00:07
elapsed_time_process:  00:00:03
df_scan file not found at ../data/datasets/logbot_data/npz_files_scan/omizunagidori/df_scan_OM1803.csv -> skipping cross-check with npz pipeline


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40318/40318 [00:34<00:00, 1156.08it/s]


window_counter: 1704
len(label_id_list): 1704
len(scanned_unixtime_list): 0
len(scanned_label_id_list): 0
scanned_unixtime_values.shape: (0,)
scanned_label_id_values.shape: (0,)
data_array.shape: (1704, 123)
----- Check label_id and scanned label_id              -----
skipped (df_scan not available)
N of extracted windows (len(df_features)): 1704
N of extracted window counter: 1704
  animal_id       unixtime       label label_id           acc_x_mean  \
0    OM1803  1536442866.96  stationary      200           0.12408208   
1    OM1803  1536442867.96  stationary      200  0.12098634000000001   
2    OM1803  1536442868.96  stationary      200  0.11856451999999998   

              acc_y_mean          acc_z_mean              acc_x_var  \
0              0.0049805  0.9279004000000001  0.0028135068260335995   
1    0.03184568000000001          1.02603518  0.0037574332431443997   
2  -0.033789099999999996  0.9284962200000001  0.0027340289487695997   

              acc_y_var             acc_z

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23536/23536 [00:30<00:00, 775.96it/s]


window_counter: 1425
len(label_id_list): 1425
len(scanned_unixtime_list): 0
len(scanned_label_id_list): 0
scanned_unixtime_values.shape: (0,)
scanned_label_id_values.shape: (0,)
data_array.shape: (1425, 123)
----- Check label_id and scanned label_id              -----
skipped (df_scan not available)
N of extracted windows (len(df_features)): 1425
N of extracted window counter: 1425
  animal_id       unixtime            label label_id  acc_x_mean  \
0    OM1804  1536268146.96  flight_cruising      401  0.27365232   
1    OM1804  1536268147.96  flight_cruising      401   0.2604395   
2    OM1804  1536268148.96  flight_cruising      401   0.2277637   

               acc_y_mean          acc_z_mean             acc_x_var  \
0   -0.057333999999999996  1.1642480199999998    0.0427585419920976   
1             -0.05440426          1.04139646      0.04446857960381   
2  -0.0024218199999999994          0.90516598  0.036479639740410005   

              acc_y_var            acc_z_var  ...     acc

...

window_counter: 648
len(label_id_list): 648
len(scanned_unixtime_list): 0
len(scanned_label_id_list): 0
scanned_unixtime_values.shape: (0,)
scanned_label_id_values.shape: (0,)
data_array.shape: (648, 123)
----- Check label_id and scanned label_id              -----
skipped (df_scan not available)
N of extracted windows (len(df_features)): 648
N of extracted window counter: 648
  animal_id       unixtime       label label_id            acc_x_mean  \
0    OM2212  1663275559.96  stationary      200  -0.09048705492424242   
1    OM2212  1663275560.96  stationary      200  -0.07800371172348484   
2    OM2212  1663275561.96  stationary      200  -0.05957434994318179   

            acc_y_mean          acc_z_mean              acc_x_var  \
0  0.09611224712121208   0.918861391439394   0.006469380447967588   
1  0.08431335077651513  0.9118920403977272  0.0035854011206756875   
2  0.13823861285984848  0.9607245469886362   0.009826567454855382   

              acc_y_var              acc_z_var  ..

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 43484/43484 [00:25<00:00, 1723.47it/s]


window_counter: 2259
len(label_id_list): 2259
len(scanned_unixtime_list): 0
len(scanned_label_id_list): 0
scanned_unixtime_values.shape: (0,)
scanned_label_id_values.shape: (0,)
data_array.shape: (2259, 123)
----- Check label_id and scanned label_id              -----
skipped (df_scan not available)
N of extracted windows (len(df_features)): 2259
N of extracted window counter: 2259
  animal_id       unixtime            label label_id             acc_x_mean  \
0    OM2213  1663273556.96  flight_cruising      401  -0.055458858049242415   
1    OM2213  1663273557.96  flight_cruising      401   -0.16241198924242425   
2    OM2213  1663273558.96  flight_cruising      401   -0.17125631028409088   

             acc_y_mean          acc_z_mean             acc_x_var  \
0  -0.22545999003787875  0.8118057329924241  0.061959155622762385   
1   -0.3267955668181818  0.9502539386742425   0.17197602930758266   
2  -0.39647894664772726  1.1855963976704544    0.1970552561549319   

             acc_y_va

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 46914/46914 [00:05<00:00, 7827.41it/s]

window_counter: 1361
len(label_id_list): 1361
len(scanned_unixtime_list): 0
len(scanned_label_id_list): 0
scanned_unixtime_values.shape: (0,)
scanned_label_id_values.shape: (0,)
data_array.shape: (1361, 123)
----- Check label_id and scanned label_id              -----
skipped (df_scan not available)
N of extracted windows (len(df_features)): 1361
N of extracted window counter: 1361
  animal_id       unixtime       label label_id           acc_x_mean  \
0    OM2214  1663447457.96  stationary      200  0.23542539460227274   
1    OM2214  1663447458.96  stationary      200  0.17443541115530298   
2    OM2214  1663447459.96  stationary      200  0.18188354160984846   

              acc_y_mean          acc_z_mean             acc_x_var  \
0   -0.12433799918560604  0.9334122472916667   0.03296080229387313   
1  -0.012131538446969696  0.9025546371969696  0.025817310761931882   
2    0.06119151954545455  0.9474044228787877  0.004972468869564612   

              acc_y_var             acc_z_var